# 03: Camada Gold: modelo dimensional

**TC3 · PosTech FIAP Data Analytics**  
Responsável: Caio Bosnic

---

**Entrada:** `silver_dw_fat_respondente`, 14.002 linhas × 79 colunas, particionada por `ano_pesquisa`.  
**Saída:** 16 tabelas em `gold/`, um star schema.

| O que | Quantas |
|---|---|
| fato, grão de respondente por edição | 1 |
| dimensões conformadas | 11 |
| catálogo de opções de múltipla escolha | 1 |
| bridge fato x opção | 1 |
| desconectadas: benchmark externo e recomendações | 2 |

As regras vivem em `glue/gold/job_gold.py` e `glue/gold/gold_comum.py`. Este
notebook não reimplementa nada: ele chama as mesmas funções que rodam no Glue
e mostra a evidência de cada etapa.

**Antes de rodar:** o notebook `02_silver.ipynb` precisa ter gerado a Silver.


In [ ]:
import os
import sys


def achar_raiz():
    """Sobe até achar a raiz do projeto, para o notebook rodar de qualquer pasta."""
    p = os.getcwd()
    for _ in range(6):
        if os.path.isfile(os.path.join(p, "glue", "gold", "job_gold.py")):
            return p
        pai = os.path.dirname(p)
        if pai == p:
            break
        p = pai
    raise RuntimeError("Não achei a raiz do projeto a partir de " + os.getcwd())


RAIZ = achar_raiz()
sys.path.insert(0, os.path.join(RAIZ, "glue", "gold"))

# Caminhos. No Glue eles vêm como Job parameter; aqui, como variável de ambiente.
os.environ.setdefault("TC3_PATH_SILVER", os.path.join(RAIZ, "data", "silver", "state_data"))
os.environ.setdefault("TC3_PATH_GOLD", os.path.join(RAIZ, "data", "gold"))

print("raiz  :", RAIZ)
print("silver:", os.environ["TC3_PATH_SILVER"])
print("gold  :", os.environ["TC3_PATH_GOLD"])


## Sessão Spark e leitura da Silver

`ler_silver` já aplica a coerção de `ano_pesquisa` para inteiro: quando a
Silver vem do Data Catalog, a coluna de partição chega como `varchar` porque
o crawler tipa pelo nome da pasta, não pelo Parquet.


In [ ]:
from gold_comum import PATH_GOLD, PATH_SILVER, ler_silver, sessao

spark = sessao("tc3-gold-notebook")
spark.sparkContext.setLogLevel("ERROR")

silver = ler_silver(spark).cache()
print("Spark", spark.version)
print(f"Silver: {silver.count()} linhas × {len(silver.columns)} colunas")
silver.groupBy("ano_pesquisa").count().orderBy("ano_pesquisa").show()


## 1. As dimensões

Cada dimensão recebe uma surrogate key determinística, um hash das colunas de
negócio. Determinística de propósito: rodar duas vezes gera a mesma chave, e
o modelo do Power BI não quebra a cada recarga.

As contagens incluem os **membros de ausência nomeados**. Em vez de deixar a FK
nula e o Power BI inventar um "(Em branco)", a Gold grava "Gestor, sem cargo
técnico", "Fora da área de dados" e "Não declarado". É por isso que
`dim_cargo` tem 20 linhas e não 18.


In [ ]:
from job_gold import construir_dimensoes

dims = construir_dimensoes(spark, silver)

print(f"{len(dims)} dimensões\n")
for nome, df in sorted(dims.items()):
    print(f"  {nome:28s} {df.count():5d} linhas")


In [ ]:
# Duas delas não descrevem respondente e ficam sem relacionamento com o fato,
# de propósito: benchmark externo e as recomendações ao cliente.
dims["dim_senioridade"].orderBy("ordem").show(truncate=False)
dims["dim_benchmark"].select("indicador", "valor_texto", "fonte", "uso").show(4, truncate=38)


## 2. O catálogo de opções e a bridge

Os 17 grupos de múltipla escolha da pesquisa não cabem no fato: uma pessoa
marca N linguagens, N clouds, N barreiras. Modelar como coluna daria uma
tabela larga e impossível de agregar.

A saída é o par catálogo mais bridge: `dim_opcao` é o dicionário de todas as
opções possíveis, e a bridge tem uma linha por respondente por opção marcada.


In [ ]:
from job_gold import construir_bridge

dim_opcao, bridge = construir_bridge(silver)

print(f"dim_opcao: {dim_opcao.count()} opções em {dim_opcao.select('grupo').distinct().count()} grupos")
print(f"bridge   : {bridge.count()} marcações\n")
dim_opcao.groupBy("grupo").count().orderBy("grupo").show(20, truncate=False)


## 3. O fato

Grão: **um respondente por edição**. As FKs apontam para as dimensões, e as
colunas `qtd_*` guardam quantas opções a pessoa marcou em cada grupo.

Essas contagens são a defesa contra o erro mais fácil desta pesquisa: `NULL`
quer dizer "não viu a pergunta" e `0` quer dizer "viu e não marcou nada".
Percentual calculado sobre a base errada inverte a conclusão.


In [ ]:
from job_gold import construir_fato

fato = construir_fato(silver, dims).cache()

print(f"fato: {fato.count()} linhas × {len(fato.columns)} colunas")
fato.select("sk_respondente", "ano_pesquisa", "sk_cargo", "sk_senioridade",
            "salario_estimado", "eh_gestor", "serie_comparavel").show(5, truncate=False)


## 4. Validação antes de gravar

`validar` confere integridade referencial de cada FK, unicidade da chave do
fato e o total por edição. Se algo não bater, para aqui: Gold errada vira
dashboard errado, e o erro só apareceria na apresentação.


In [ ]:
from job_gold import validar

validar(fato, dims, dim_opcao, bridge)


## 5. Gravar as 16 tabelas


In [ ]:
from gold_comum import gravar
from job_gold import PREFIXO

gravar(fato, PREFIXO + "fat_profissionais", particao="ano_pesquisa", mostrar=False)
for nome, df in dims.items():
    gravar(df, PREFIXO + nome, mostrar=False)
gravar(dim_opcao, PREFIXO + "dim_opcao", mostrar=False)
gravar(bridge, PREFIXO + "bridge_respondente_opcao", mostrar=False)


## 6. Conferência final

Lê de volta o que foi gravado em disco, não o DataFrame em memória: é o
Parquet que o Athena e o Power BI vão consumir.


In [ ]:
gravadas = sorted(d for d in os.listdir(PATH_GOLD)
                  if os.path.isdir(os.path.join(PATH_GOLD, d)))
print(f"{len(gravadas)} tabelas em {PATH_GOLD}\n")
for t in gravadas:
    n = spark.read.parquet(os.path.join(PATH_GOLD, t)).count()
    print(f"  {t:38s} {n:7d} linhas")

assert len(gravadas) == 16, f"esperado 16 tabelas, vieram {len(gravadas)}"
print("\n[OK] as 16 tabelas da Gold estão em disco")


---

Próximo: [`04_analises_e_graficos.ipynb`](04_analises_e_graficos.ipynb), que
responde as sete perguntas em cima destas tabelas e gera os gráficos.

O modelo está descrito em [`docs/MODELO_GOLD.md`](../docs/MODELO_GOLD.md).
